In [ ]:
### =============================================###
###                 Manager Agent                ###
### =============================================###

# ⚠️ This is the Manager Agent (Foundation LLM: DeepSeek-Chat).
# ⚠️ Please make sure to replace the API key with your own one.

from openai import OpenAI
client = OpenAI(api_key="your_own_api", base_url="https://api.deepseek.com")

# Example of user input query (text or voice)
user_query = f"""
Simulate a 1 story hollow square, courtyard building.
The height of story 1 is 11.42 meters.
The horizontal segments are 144.24 meters long and 19.33 meters wide.
The vertical segments are 29.80 meters long and 390.88 meters wide.
The attic height is 26.60 meters. The building orientation is 352 degrees to the north.
Each story has 4 thermal zones in each orientation.
The window-to-wall ratio is 0.80 for the north, 0.58 for the south, 0.17 for the west, and 0.78 for the east.

The wall is made of concrete, with a thickness of 0.03 meters and the wall insulation is R18.
The roof is made of concrete, with a thickness of 0.05 meters and the roof insulation is R18.
The floor is made of concrete, covered with carpet.
The window U-factor is 1.8 W/m2K and the SHGC is 0.3.
This building has 1 space types. Space type 1 is for zone 1, 2, 3, 4. For space type 1, the people density is 10 m2/person, the lighting density is 4 W/m2, and the electric equipment density is 8 W/m2.
The people activity level is 130 W/person. The infiltration rate is 0.11 ACH.
The occupancy rate is 0.9 from 8:00 to 16:00 and 0.2 in other periods of time.
For zone 1, 2, 3, 4, the cooling setpoint is 24.0 Celsius during 6:00 to 18:00, and 26.0 Celsius in unoccupied periods. The heating setpoint is 20.0 Celsius during 6:00 to 18:00, and 18.0 Celsius in unoccupied periods.

The HVAC system in this building is packaged air conditioning unit, rooftop unit, DX system with electric heater for heating.
The unit 1 serves zone 1, zone 2.
It includes an economizer, which operates based on differential dry bulb.
The rated capacity is 41378 W for cooling and Autosize W for heating.
The rated cooling COP is 3.8 and the heating efficiency is 0.75.
The supply air temperature for cooling is 14.3 Celsius, and for heating is 40.7 Celsius.
The outdoor ventilation rate is 3.69 ACH.
The fan efficiency is 0.78, the pressure rise is 1770 Pa, and the maximum flow rate is Autosize m3/s.
The unit 2 serves zone 3, zone 4.
The rated capacity is Autosize W for cooling and 36963 W for heating.
The rated cooling COP is 3.96 and the heating efficiency is 0.87.
The supply air temperature for cooling is 18.6 Celsius, and for heating is 49.0 Celsius.
The outdoor ventilation rate is 1.18 ACH.
The fan efficiency is 0.64, the pressure rise is 1536 Pa, and the maximum flow rate is 130.08 m3/s.
"""

# Manager's prompt
manager_prompt = f"""
Please extract sentences related to the following categories from the provided content:
1. Building geometry, including shape, story, height, orientation, window-to-wall ratio, etc.
2. Building information, including materials, construction, insulations, information of space types, cooling and heating setpoints.
3. HVAC systems, including fan and coil units, control methods, supply temperature, ventilation, economizer, outdoor air system, etc.
4. Water loops, including chilled water, hot water, condenser water, and related equipment.

For each category, identify and extract only the relevant sentences.

The output format:
**Category 1:** (the building geometry sentences).
**Category 2:** (the building information sentences).
**Category 3:** (the HVAC system sentences).
**Category 4:** (the water loop sentences).

Provided content:
"""

response = client.chat.completions.create(
    model="deepseek-chat",
    messages=[
        {"role": "system", "content": manager_prompt},
        {"role": "user", "content": user_query},
    ],
    stream=False,
)

print(response.choices[0].message.content)

# An extraction function, extract_categories, is designed to extract different categories based on the regular expression **Category {n}:** defined in the manager_prompt.
# category_1, category_2, category_3, category_4 = extract_categories(response.choices[0].message.content)

### =============================================###
###           Fine-Tuned Modeler Agents          ###
### =============================================###

# ⚠️ Please make sure you have adequate GPU memory.
! pip install -U bitsandbytes -q # pip this repo at your first run
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
import torch
from peft import PeftModel, PeftConfig
import os

os.environ["CUDA_VISIBLE_DEVICES"] = "0"
device = "cuda:0"

# Load the Modeler config.
repo_id = "GangJiang/LLM-BEM-Engineer"
sub = "Modeler-Bldg_Geo" # 4 related agent names: Modeler-Bldg_Geo; Modeler-Bldg_Cons_Ops; Modeler-Water_Sys; Modeler-Air-Sys
config = PeftConfig.from_pretrained(repo_id, subfolder=sub)

# Load the base LLM, flan-t5-xl (8-bit quantized), and tokenizer
base_model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-xl", load_in_8bit=True)
tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-xl")

# Load the Lora model
modeler_agent_model = PeftModel.from_pretrained(base_model, repo_id, subfolder=sub)
# Generation config
generation_config = modeler_agent_model.generation_config
generation_config.max_new_tokens = 3000
generation_config.temperature = 0.1
generation_config.top_p = 0.1
generation_config.num_return_sequences = 1
generation_config.pad_token_id = tokenizer.eos_token_id
generation_config.eos_token_id = tokenizer.eos_token_id

# This is the corresponding Modeler Agent input extract from the Manager Agent (details in the file: "Manager_Agent_Inference.ipynb").
# For example, modeler agent: Modeler-Bldg_Geo

modeler_agent_input=f"""
Simulate a 1 story hollow square, courtyard building.
The height of story 1 is 11.42 meters.
The horizontal segments are 144.24 meters long and 19.33 meters wide.
The vertical segments are 29.80 meters long and 390.88 meters wide.
The attic height is 26.60 meters.
The building orientation is 352 degrees to the north.
Each story has 4 thermal zones in each orientation.
The window-to-wall ratio is 0.80 for the north, 0.58 for the south, 0.17 for the west, and 0.78 for the east.
"""
modeler_agent_model.to(device)

# Modeler Agent generating...
input_ids = tokenizer(modeler_agent_input, return_tensors="pt", truncation=False).to(device)
generated_ids = modeler_agent_model.generate(input_ids = input_ids.input_ids,
                           attention_mask = input_ids.attention_mask,
                           generation_config = generation_config)
generated_output = tokenizer.decode(generated_ids[0], skip_special_tokens=True)

# View the generated result of the fine-tuned corresponding Modeler (e.g., Modeler-Bldg_Geo)
agent_action = generated_output
print(agent_action)

# Execute the modeling actions produced by the Modeler Agent
# exec(agent_action)